[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%205/L9_Demo_DrugReviewAssistant.ipynb)

# ISBA 2411 · Week 5 · Lecture 9
## Standing on Giants — Hugging Face Sentiment & QA

**The Drug-Review Assistant.** Last week you *built* a neural network from scratch. Tonight you *borrow* one: a model someone already trained on billions of words, that gives you **sentiment analysis** and **question answering** over 215K real drug reviews in three lines of code.

By the end of this notebook you'll have a live **chat app** the whole class can use — and you'll spend the demo trying to *break* it, because the real skill isn't calling the model, it's **judging** it.

---
**Runtime:** Colab (free CPU is fine; GPU optional). Run the cells top to bottom.

## 0 · Setup
We pin `transformers` to the 4.x line so the classic `pipeline("question-answering")` works, and install Gradio for the chat UI.
> If Colab prints *"You must restart the runtime"* after this cell, click **Restart**, then run it again and continue.

In [ ]:
%pip install -q "transformers>=4.40,<5" "gradio>=4.0" scikit-learn pandas

import pandas as pd, re, html, textwrap
from transformers import pipeline
print('setup complete')

## 1 · Load and clean the reviews
The raw Drugs.com data has HTML entities (`&amp;`), wrapper quotes, and ~1,800 scraping-artifact rows. We load a pre-sampled 8,000-review slice and tidy the text.

*If the URL fails (e.g., the repo file isn't published yet), the cell falls back to a Colab file upload — just pick `drug_reviews_demo.csv`.*

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/drug_reviews_demo.csv'

def clean_text(t):
    t = html.unescape(str(t)); t = t.strip().strip('"').strip()
    return re.sub(r'\s+', ' ', t)

try:
    df = pd.read_csv(DATA_URL)
    print('loaded from URL')
except Exception as e:
    print('URL load failed -> upload the CSV.  (', e, ')')
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(next(iter(up)))

df['review'] = df['review'].map(clean_text)
df = df[df['review'].str.len() > 40].reset_index(drop=True)
print(f'{len(df):,} reviews | {df.drugName.nunique()} drugs | {df.condition.nunique()} conditions')
df.head(3)

## 2 · Task 1 — Sentiment analysis in three lines
A `pipeline` loads a pretrained model **and** does all the plumbing (tokenize -> run -> decode). This one was trained on movie reviews (SST-2); we point it at *drug* reviews and see how far the borrowed knowledge transfers.

In [ ]:
sentiment = pipeline('sentiment-analysis',
                     model='distilbert-base-uncased-finetuned-sst-2-english')

for r in df['review'].sample(3, random_state=7):
    out = sentiment(r[:512])[0]
    print(f"{out['label']:8} {out['score']:.2f}  |  {textwrap.shorten(r, 90)}")

### The catch: confident **and** wrong
The model returns a confidence for *every* input — including the ones it gets wrong. Watch what negation and mild language do to it. **This is the whole lesson:** your judgment is the safeguard.

In [ ]:
tricky = [
    'No side effects at all, I feel completely normal.',      # negation
    'Great. Another pill that does absolutely nothing.',       # sarcasm
    'It works but the nausea is not worth it.',                # mixed
    'I stopped taking it after two days.',                     # neutral-ish
]
for t in tricky:
    o = sentiment(t)[0]
    print(f"{o['label']:8} {o['score']:.2f}  |  {t}")

## 3 · Task 2 — Extractive Question Answering
Given a **context** (some review text) and a **question**, the model *points to the answer span inside the text*. It doesn't invent an answer — it extracts one. That's a feature (no hallucination) and a limit (it can only return words that are already there).

In [ ]:
qa = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')

context = ('I took Lexapro for anxiety. It worked well after two weeks '
           'but caused drowsiness and some nausea at first.')
for q in ['What was it taken for?', 'What side effects did they have?']:
    a = qa(question=q, context=context)
    print(f"Q: {q}\n   A: {a['answer']!r}  (score {a['score']:.2f})\n")

## 4 · Give QA the *right* reviews to read (Week-4 callback)
QA needs good context. We reuse **TF-IDF retrieval** from the retrieval hackathon to pull the reviews most relevant to a question, then feed those in as context. Keyword search picks *what to read*; QA does *the reading*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vec = TfidfVectorizer(stop_words='english', max_features=5000)
M = vec.fit_transform(df['review'])

def top_reviews(query, k=3):
    sims = linear_kernel(vec.transform([query]), M).ravel()
    return df['review'].iloc[sims.argsort()[::-1][:k]].tolist()

q = 'What are common side effects?'
ctx = ' '.join(top_reviews(q, k=3))[:2000]
print(qa(question=q, context=ctx))

## 5 · Wire it into one assistant
Simple routing: a **question** -> retrieve + QA;  anything else (a pasted review, *"is this positive?"*) -> sentiment. This function is the brain behind the chat box.

In [ ]:
QUESTION_STARTS = {'what','why','how','is','are','does','do','can','should','which','when','who'}

def looks_like_question(text):
    t = text.strip().lower()
    return t.endswith('?') or (t.split() and t.split()[0] in QUESTION_STARTS)

def assistant(message, history=None):
    if looks_like_question(message):
        ctx = ' '.join(top_reviews(message, k=3))[:2000]
        a = qa(question=message, context=ctx)
        return f"**Answer (from reviews):** {a['answer']}  \n_confidence {a['score']:.0%}_"
    o = sentiment(message[:512])[0]
    return f"**Sentiment:** {o['label']}  \n_confidence {o['score']:.0%}_"

print(assistant('What side effects do people report?'))
print(assistant('This medication changed my life for the better.'))

## 6 · The chat app — the whole class joins
`launch(share=True)` prints a **public URL** (good for ~72 hours). Put it on screen; students open it on a laptop or phone and start chatting. **Your mission during the demo:**
1. Find a review where the **sentiment** is wrong — and say *why* (negation? sarcasm? mixed?).
2. Ask a question it answers well, and one it **fumbles** — name the failure mode.


In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=assistant,
    title='Drug-Review Assistant  ·  ISBA 2411',
    description='Ask a question about the reviews, or paste a review to get its sentiment. '
                'Then try to break it.',
    examples=['What side effects do people report?',
              'Is this drug good for anxiety?',
              'No side effects at all, I feel completely normal.',
              'This medication ruined my life.'],
)
demo.launch(share=True)

### Fallback (if campus wi-fi blocks the public link)
Run this instead for an inline chat box — same brain, no public URL.

In [ ]:
import ipywidgets as w
from IPython.display import display, Markdown
box = w.Text(placeholder='Ask a question or paste a review...', layout=w.Layout(width='70%'))
send = w.Button(description='Send', button_style='primary'); out = w.Output()
def handle(_):
    if not box.value.strip():
        return
    with out: display(Markdown(f"**You:** {box.value}\n\n{assistant(box.value)}\n\n---"))
    box.value = ''
send.on_click(handle)          # button works on every ipywidgets version
display(w.HBox([box, send]), out)

---
## Take-home exercise
1. Run the sentiment pipeline on **20 reviews** and compare its label to the star `rating` (call 1-4 negative, 7-10 positive). How often do they agree?
2. Collect **three failure cases** (sentiment or QA) and write a one-line diagnosis for each (negation / sarcasm / mixed / out-of-domain question / wrong context retrieved).
3. **Stretch:** swap in a different Hugging Face model (e.g., a review-tuned sentiment model) and see whether the failures change.

Hand in a short write-up: your approach, what you found, and the three failure cases. *(This maps straight to the midterm rubric's Analysis & Insight.)*